### use plos-one data to regress the relationship between author demographic information, contribution and authorship order

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import copy
import itertools
import networkx as nx
import random
from collections import defaultdict
import matplotlib.pyplot as plt
import multiprocessing
from functools import partial
import seaborn as sns

In [2]:
plos_path = "../../data/dropbox/LaborDivision/data/PloS-credit/PLOS Credit Full.xlsx"

In [3]:
sheet1 = pd.read_excel(plos_path, sheet_name="widget (70)",
                      names=['status','doi','author_cnt','journal','credit','name','author_order','gender','author_role','nan1','nan2'])
sheet1 = sheet1.drop(columns=['nan1','nan2'])

### read ostbk number for the plos data

In [4]:
plos_ostbk_path = '../../data/dropbox/LaborDivision/data/PloS-credit/plos_ostbk.txt'
plos_ostbk = pd.read_csv(plos_ostbk_path, sep=",",names=['ost_bk','doi'],header=1)
plos_ostbk.head(2)

,ost_bk,doi
0,66915750.0,10.1371/journal.pone.0189230
1,78127338.0,10.1371/journal.pone.0180085


In [49]:
data = plos_ostbk.merge(sheet1.drop(columns=['status','author_cnt','journal','gender','author_role']),
                        on='doi',
                       how='left')
total_plos = data['doi'].nunique() #total number of articles in plost dataset
data = data.drop(columns=['doi'])

### remove data without ost_bk and author_order number

In [50]:
invalid_id = data[data.ost_bk.isna()|data.author_order.isna()]['ost_bk'].unique()
data = data[~data.ost_bk.isin(invalid_id)]

### remove rows that are duplicated

In [51]:
data = data.drop_duplicates(subset=['ost_bk','author_order','credit'])

### read author information from existing file

In [11]:
all_df = pd.read_csv("../../data/dropbox/LaborDivision/data/deriveddata/all/pub_author.csv")
all_df = all_df.drop_duplicates(subset=['ost_bk','cluster_id'])

In [12]:
all_df.head(2)

,ost_bk,seqno,cluster_id,lastname,cntry,num_author,role,year,dis,title,alpha,intl
0,29768068,1.0,40806321.0,Miron,CAN,5.0,F,2008,56,Cyclical and dose-dependent responses of adult...,noalp,0
1,29768068,2.0,41639269.0,Hall,CAN,5.0,M,2008,56,Cyclical and dose-dependent responses of adult...,noalp,0


### merge author information with plos data

In [52]:
data = data.merge(all_df, left_on=['ost_bk','author_order'], right_on=['ost_bk','seqno'], how='left')
data = data[['ost_bk','year','dis','alpha','cluster_id','seqno','cntry','credit','role']]

### remove publications that are not included in the all data. Those publications are removed in the data cleaning process due to their incomplete author information (confirmed by sampling)

In [53]:
invalid_ids = data[data.seqno.isna()]['ost_bk'].unique()
data = data[~data.ost_bk.isin(invalid_ids)]
print("number of papers are removed %s"%(len(invalid_ids)))

number of papers are removed 4543


### merge the cntry with the st code

In [43]:
st_group = pd.read_csv("/u/miaoli/ember_home/labordivision/data/dropbox/LaborDivision/data/additionaldata/cntry_classification/cntry_st.csv")
st_group = st_group.drop_duplicates()
st_group.head(2)

,Code,class
0,AFG,Lagging
1,ALB,Lagging


In [54]:
data = data.merge(st_group, left_on='cntry', right_on='Code')
data.sample(1)

,ost_bk,year,dis,alpha,cluster_id,seqno,cntry,credit,role,Code,class
454311,51456724.0,2018.0,18.0,noalp,79707382.0,1.0,DEU,Methodology,F,DEU,Advanced


### select publications that are collaborated between different groups

In [55]:
data['intl'] = data.groupby(['ost_bk'])['class'].transform('nunique')
data = data[data['intl']>1]

### read author demographic information

In [47]:
author_info_path='../../data/dropbox/LaborDivision/data/cleanedraw/author_info.pkl'
author_info = pd.read_pickle(author_info_path)
author_info.head(2)

,cluster_id,gender,yfp,npub
0,81178534.0,F,2012.0,11.0
1,4745329.0,F,2013.0,746.0


In [57]:
data = data.merge(author_info, on='cluster_id', how='left')

### convert number of publication to log

In [62]:
data['npub'] = np.log(data['npub'])

In [65]:
data.to_csv("plos_regression.csv", index=False)

In [68]:
data[data.alpha=='noalp']['ost_bk'].nunique()

4084

In [70]:
data[data.ost_bk==65855275]

,ost_bk,year,dis,alpha,cluster_id,seqno,cntry,credit,role,Code,class,intl,gender,yfp,npub
0,65855275.0,2018.0,64.0,noalp,83846425.0,4.0,ESP,Investigation,M,ESP,Proficient,3,M,2007.0,4.317488
1,65855275.0,2018.0,64.0,noalp,83846425.0,4.0,ESP,Writing - Original Draft,M,ESP,Proficient,3,M,2007.0,4.317488
2,65855275.0,2018.0,64.0,noalp,83846425.0,4.0,ESP,Writing - Review & Editing,M,ESP,Proficient,3,M,2007.0,4.317488
20335,65855275.0,2018.0,64.0,noalp,20221544.0,7.0,USA,Funding acquisition,M,USA,Advanced,3,U,2005.0,4.369448
20336,65855275.0,2018.0,64.0,noalp,20221544.0,7.0,USA,Investigation,M,USA,Advanced,3,U,2005.0,4.369448
20337,65855275.0,2018.0,64.0,noalp,20221544.0,7.0,USA,Methodology,M,USA,Advanced,3,U,2005.0,4.369448
20338,65855275.0,2018.0,64.0,noalp,20221544.0,7.0,USA,Writing - Original Draft,M,USA,Advanced,3,U,2005.0,4.369448
20339,65855275.0,2018.0,64.0,noalp,20221544.0,7.0,USA,Writing - Review & Editing,M,USA,Advanced,3,U,2005.0,4.369448
47345,65855275.0,2018.0,64.0,noalp,37270295.0,6.0,GBR,Writing - Original Draft,M,GBR,Advanced,3,M,2010.0,3.761200
47346,65855275.0,2018.0,64.0,noalp,37270295.0,6.0,GBR,Writing - Review & Editing,M,GBR,Advanced,3,M,2010.0,3.761200
